In [1]:
# Cell 1 — Install dependencies
!pip install torch-geometric

In [2]:
# Cell 2 — Imports
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("All imports successful")
print(f"PyTorch version: {torch.__version__}")

c:\Users\sohan\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports successful
PyTorch version: 2.12.0+cpu


In [3]:
# Cell 3 — Define the Physics Graph Structure
# This is the core of your contribution
# The engine gas path: Fan → LPC → HPC → HPT → LPT

# 5 nodes = 5 engine components
# Node indices: 0=Fan, 1=LPC, 2=HPC, 3=HPT, 4=LPT

COMPONENTS = ['Fan', 'LPC', 'HPC', 'HPT', 'LPT']

# Edges follow physical gas path direction only
# Fan affects LPC, LPC affects HPC, HPC affects HPT, HPT affects LPT
edge_index = torch.tensor([
    [0, 1, 2, 3],   # source nodes
    [1, 2, 3, 4]    # target nodes
], dtype=torch.long)

print("Physics Graph defined:")
print(f"Nodes: {COMPONENTS}")
print(f"Edges (physical connections):")
print(f"  Fan → LPC")
print(f"  LPC → HPC")
print(f"  HPC → HPT")
print(f"  HPT → LPT")
print(f"Edge index shape: {edge_index.shape}")

Physics Graph defined:
Nodes: ['Fan', 'LPC', 'HPC', 'HPT', 'LPT']
Edges (physical connections):
  Fan → LPC
  LPC → HPC
  HPC → HPT
  HPT → LPT
Edge index shape: torch.Size([2, 4])


In [4]:
# Cell 4 — Generate the same synthetic data as the mentor's notebook

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

N_TRAIN_ENGINES = 6
N_TEST_ENGINES = 3
MAX_CYCLES = 350
MIN_CYCLES = 150
WINDOW = 30
MAX_RUL = 125
FAULT_MODE = 'HPC'  # HPC component is degrading

# Sensor to component influence matrix (14 sensors x 5 components)
# Which sensors are affected by which components
SENSOR_COMP_INFLUENCE = np.array([
    [0.8, 0.1, 0.0, 0.0, 0.0],  # T24  - Fan outlet temp
    [0.1, 0.7, 0.2, 0.0, 0.0],  # T30  - HPC outlet temp
    [0.0, 0.1, 0.8, 0.1, 0.0],  # T48  - HPT outlet temp
    [0.0, 0.0, 0.1, 0.8, 0.1],  # T50  - LPT outlet temp
    [0.6, 0.2, 0.0, 0.0, 0.0],  # P15  - Fan inlet pressure
    [0.5, 0.1, 0.0, 0.0, 0.0],  # P2   - Fan inlet pressure
    [0.3, 0.5, 0.1, 0.0, 0.0],  # P21  - Fan outlet pressure
    [0.0, 0.2, 0.7, 0.1, 0.0],  # P24  - HPC inlet pressure
    [0.0, 0.1, 0.8, 0.1, 0.0],  # Ps30 - HPC outlet pressure
    [0.0, 0.0, 0.2, 0.7, 0.1],  # P40  - HPT outlet pressure
    [0.0, 0.0, 0.1, 0.2, 0.7],  # P50  - LPT outlet pressure
    [0.7, 0.2, 0.1, 0.0, 0.0],  # Nf   - Fan speed
    [0.1, 0.2, 0.6, 0.1, 0.0],  # Nc   - Core speed
    [0.0, 0.1, 0.5, 0.3, 0.1],  # Wf   - Fuel flow
])

SENSOR_NAMES = ['T24','T30','T48','T50','P15','P2',
                'P21','P24','Ps30','P40','P50','Nf','Nc','Wf']

def generate_engine_data(n_engines, max_cycles, min_cycles, fault_mode='HPC'):
    all_X, all_Y, all_H = [], [], []
    fault_comp = COMPONENTS.index(fault_mode)
    
    for eng in range(n_engines):
        cycles = np.random.randint(min_cycles, max_cycles)
        t = np.arange(cycles)
        
        # Health parameters — only fault component degrades significantly
        health = np.ones((cycles, 5))
        degradation = (t / cycles) ** 1.5
        health[:, fault_comp] = 1.0 - 0.8 * degradation
        
        # Add some noise to other components
        for c in range(5):
            if c != fault_comp:
                health[:, c] += np.random.normal(0, 0.02, cycles)
        health = np.clip(health, 0, 1)
        
        # Sensor readings based on health params and physics
        X = np.zeros((cycles, 14))
        for s in range(14):
            base = SENSOR_COMP_INFLUENCE[s] @ health.T
            noise = np.random.normal(0, 0.02, cycles)
            X[:, s] = base + noise
        
        # RUL
        Y = np.maximum(0, cycles - t - 1)
        Y = np.minimum(Y, MAX_RUL)
        
        all_X.append(X)
        all_Y.append(Y)
        all_H.append(health)
    
    return all_X, all_Y, all_H

# Generate data
train_X, train_Y, train_H = generate_engine_data(N_TRAIN_ENGINES, MAX_CYCLES, MIN_CYCLES)
test_X, test_Y, test_H = generate_engine_data(N_TEST_ENGINES, MAX_CYCLES, MIN_CYCLES)

print(f"Train engines: {len(train_X)}")
print(f"Test engines: {len(test_X)}")
print(f"Sample engine cycles: {len(train_X[0])}")
print(f"Sensors per cycle: {train_X[0].shape[1]}")
print("Data generation complete")

Train engines: 6
Test engines: 3
Sample engine cycles: 252
Sensors per cycle: 14
Data generation complete


In [5]:
# Cell 5 — Create sliding windows and prepare data for GNN

def make_windows(X_list, Y_list, H_list, window=WINDOW):
    windows, labels, health_windows = [], [], []
    
    for X, Y, H in zip(X_list, Y_list, H_list):
        for i in range(len(X) - window):
            windows.append(X[i:i+window])        # (window, 14 sensors)
            labels.append(Y[i+window-1])          # RUL at end of window
            health_windows.append(H[i+window-1]) # health params at end
    
    return (np.array(windows), 
            np.array(labels, dtype=np.float32),
            np.array(health_windows, dtype=np.float32))

# Create windows
X_train, Y_train, H_train = make_windows(train_X, train_Y, train_H)
X_test, Y_test, H_test = make_windows(test_X, test_Y, test_H)

# Normalise RUL to 0-1
Y_train_norm = Y_train / MAX_RUL
Y_test_norm = Y_test / MAX_RUL

# Normalise sensors
scaler = StandardScaler()
X_train_flat = X_train.reshape(-1, 14)
X_train_norm = scaler.fit_transform(X_train_flat).reshape(X_train.shape)
X_test_flat = X_test.reshape(-1, 14)
X_test_norm = scaler.transform(X_test_flat).reshape(X_test.shape)

print(f"Train windows: {X_train_norm.shape}")
print(f"Test windows: {X_test_norm.shape}")
print(f"Health params shape: {H_train.shape}")
print("Preprocessing complete")

Train windows: (1465, 30, 14)
Test windows: (848, 30, 14)
Health params shape: (1465, 5)
Preprocessing complete


In [6]:
# Cell 6 — Build the Physics Graph GNN (Arm C) — FIXED

class PhysicsGNN(nn.Module):
    def __init__(self, n_sensors=14, hidden=64, n_concepts=5):
        super().__init__()
        
        # Project node features (14) to hidden size (64)
        self.input_proj = nn.Linear(n_sensors, hidden)
        
        # Graph convolution layers
        self.conv1 = GCNConv(hidden, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        
        # Each node produces a concept score
        self.concept_head = nn.Linear(hidden, 1)
        
        # Final RUL prediction from concept scores
        self.rul_head = nn.Sequential(
            nn.Linear(n_concepts, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        
        self.n_concepts = n_concepts
        
        self.register_buffer('M', torch.tensor(
            SENSOR_COMP_INFLUENCE, dtype=torch.float32
        ))
    
    def forward(self, x, edge_index, batch=None):
        # Project input to hidden size
        h = F.relu(self.input_proj(x))
        
        # Graph convolution
        h = F.relu(self.conv1(h, edge_index))
        h = F.dropout(h, p=0.2, training=self.training)
        h = F.relu(self.conv2(h, edge_index))
        
        # Concept scores per node
        concepts = torch.sigmoid(self.concept_head(h))
        
        # Reshape to (batch, 5)
        if batch is not None:
            n_graphs = batch.max().item() + 1
            concepts = concepts.view(n_graphs, self.n_concepts)
        else:
            concepts = concepts.view(-1, self.n_concepts)
        
        # Predict RUL
        rul = self.rul_head(concepts)
        
        return rul, concepts

print("PhysicsGNN architecture defined — fixed")

PhysicsGNN architecture defined — fixed


In [7]:
# Cell 7 — Prepare data for GNN (convert windows to graph format)

def prepare_gnn_data(X_norm, Y_norm, H, edge_index, M):
    """
    Convert sliding windows into graph data.
    Each sample becomes a graph with 5 nodes (components).
    Each node gets features = sensor readings projected via physics matrix.
    """
    dataset = []
    M_np = M.numpy()
    
    for i in range(len(X_norm)):
        # X_norm[i] shape: (window, 14 sensors)
        # Average over time window to get per-sensor summary
        sensor_avg = X_norm[i].mean(axis=0)  # (14,)
        
        # Project sensors to components using physics matrix
        # M shape: (14 sensors, 5 components)
        # node_features shape: (5 components, 14 sensors)
        node_features = (M_np.T * sensor_avg).astype(np.float32)  # (5, 14)
        
        # Create graph data object
        data = Data(
            x=torch.tensor(node_features, dtype=torch.float32),
            edge_index=edge_index,
            y=torch.tensor([Y_norm[i]], dtype=torch.float32),
            health=torch.tensor(H[i], dtype=torch.float32)
        )
        dataset.append(data)
    
    return dataset

# Convert to graph dataset
M_tensor = torch.tensor(SENSOR_COMP_INFLUENCE, dtype=torch.float32)

train_dataset = prepare_gnn_data(X_train_norm, Y_train_norm, H_train, edge_index, M_tensor)
test_dataset = prepare_gnn_data(X_test_norm, Y_test_norm, H_test, edge_index, M_tensor)

# Create dataloaders
from torch_geometric.loader import DataLoader as GeoDataLoader

train_loader = GeoDataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = GeoDataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Train graph samples: {len(train_dataset)}")
print(f"Test graph samples: {len(test_dataset)}")
print(f"Node features per node: {train_dataset[0].x.shape}")
print(f"Edges: {train_dataset[0].edge_index.shape}")
print("GNN data preparation complete")

Train graph samples: 1465
Test graph samples: 848
Node features per node: torch.Size([5, 14])
Edges: torch.Size([2, 4])
GNN data preparation complete


In [8]:
# Cell 8 — Train the Physics GNN

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {DEVICE}")

# Initialise model
model_c = PhysicsGNN(
    n_sensors=14,
    hidden=64,
    n_concepts=5
).to(DEVICE)

# Training setup
optimizer = torch.optim.Adam(model_c.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
rul_loss_fn = nn.MSELoss()
concept_loss_fn = nn.MSELoss()
ALPHA = 0.25

# Training loop
best_val_loss = float('inf')
patience_counter = 0
PATIENCE = 10
history = {'train_loss': [], 'val_loss': []}

print("Training Physics GNN (Arm C)...")
print(f"Concept supervision weight α = {ALPHA}")
print()

for epoch in range(80):
    # Training
    model_c.train()
    train_losses = []
    
    for batch in train_loader:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()
        
        rul_pred, concepts = model_c(batch.x, batch.edge_index, batch.batch)
        
        # RUL loss
        loss_rul = rul_loss_fn(rul_pred.squeeze(), batch.y.squeeze())
        
        # Concept supervision loss
        n_graphs = batch.batch.max().item() + 1
        concept_targets = (1 - batch.health).view(n_graphs, 5)
        loss_concept = concept_loss_fn(concepts, concept_targets)
        
        # Combined loss
        loss = ALPHA * loss_concept + (1 - ALPHA) * loss_rul
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_c.parameters(), 1.0)
        optimizer.step()
        train_losses.append(loss.item())
    
    # Validation
    model_c.eval()
    val_losses = []
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(DEVICE)
            rul_pred, concepts = model_c(batch.x, batch.edge_index, batch.batch)
            loss_rul = rul_loss_fn(rul_pred.squeeze(), batch.y.squeeze())
            val_losses.append(loss_rul.item())
    
    train_loss = np.mean(train_losses)
    val_loss = np.mean(val_losses)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    
    scheduler.step(val_loss)
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model_c.state_dict(), 'best_model_c.pt')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/80 — train: {train_loss:.5f} val: {val_loss:.5f}")

# Load best model
model_c.load_state_dict(torch.load('best_model_c.pt'))
print("\nTraining complete. Best model loaded.")

Training on: cpu
Training Physics GNN (Arm C)...
Concept supervision weight α = 0.25

Epoch 10/80 — train: 0.06652 val: 0.08468
Epoch 20/80 — train: 0.01400 val: 0.01273
Epoch 30/80 — train: 0.00801 val: 0.00896
Epoch 40/80 — train: 0.00613 val: 0.00788
Epoch 50/80 — train: 0.00486 val: 0.00674
Epoch 60/80 — train: 0.00404 val: 0.00599
Epoch 70/80 — train: 0.00331 val: 0.00643
Epoch 80/80 — train: 0.00267 val: 0.00570

Training complete. Best model loaded.


In [9]:
# Cell 9 — Evaluate Physics GNN and compute PC Score

model_c.eval()
all_preds = []
all_trues = []
all_concepts = []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(DEVICE)
        rul_pred, concepts = model_c(batch.x, batch.edge_index, batch.batch)
        all_preds.append(rul_pred.squeeze().cpu().numpy())
        all_trues.append(batch.y.squeeze().cpu().numpy())
        all_concepts.append(concepts.cpu().numpy())

pred_c = np.concatenate(all_preds) * MAX_RUL
true_c = np.concatenate(all_trues) * MAX_RUL
concepts_c = np.concatenate(all_concepts)

# RMSE and MAE
rmse_c = np.sqrt(mean_squared_error(true_c, pred_c))
mae_c = mean_absolute_error(true_c, pred_c)
print(f"Arm C (Physics GNN)")
print(f"RMSE: {rmse_c:.2f} cycles")
print(f"MAE:  {mae_c:.2f} cycles")

# PC Score
from scipy.stats import spearmanr

fault_comp = COMPONENTS.index(FAULT_MODE)
e_phys = SENSOR_COMP_INFLUENCE[:, fault_comp]
e_phys = e_phys / e_phys.sum()

# Project concepts back to sensor space via physics matrix
M_np = SENSOR_COMP_INFLUENCE
attr_c = (M_np @ concepts_c.T).T  # (N, 14)

# Normalise
attr_c_norm = attr_c / (attr_c.sum(axis=1, keepdims=True) + 1e-10)

# rho_phys
rhos = [spearmanr(attr_c_norm[i], e_phys)[0] for i in range(len(attr_c_norm))]
rho_mean = np.mean(rhos)
rho_std = np.std(rhos)

# delta_cons
pairs = [(s, s2) for s in range(14) for s2 in range(14)
         if abs(e_phys[s] - e_phys[s2]) > 0.05]
violations = []
for i in range(len(attr_c_norm)):
    v = sum(1 for s, s2 in pairs
            if (e_phys[s] > e_phys[s2]) != (attr_c_norm[i,s] > attr_c_norm[i,s2]))
    violations.append(v / len(pairs) if pairs else 0)
delta_cons = np.mean(violations)

# sigma_stab — GNN is deterministic so 0
sigma_stab = 0.0

# PC score
pc_c = rho_mean * (1 - delta_cons) * (1 - sigma_stab)

print(f"\nPC Score breakdown:")
print(f"  rho_phys:   {rho_mean:.4f} ± {rho_std:.4f}")
print(f"  delta_cons: {delta_cons:.4f}")
print(f"  sigma_stab: {sigma_stab:.6f}")
print(f"  PC (full):  {pc_c:.4f}")

print(f"\n--- Final Comparison ---")
print(f"Arm A (CNN-LSTM): PC = -0.016")
print(f"Arm B (CBM):      PC = +0.130")
print(f"Arm C (GNN):      PC = {pc_c:.4f}")

Arm C (Physics GNN)
RMSE: 8.98 cycles
MAE:  7.08 cycles

PC Score breakdown:
  rho_phys:   0.4756 ± 0.5756
  delta_cons: 0.2331
  sigma_stab: 0.000000
  PC (full):  0.3647

--- Final Comparison ---
Arm A (CNN-LSTM): PC = -0.016
Arm B (CBM):      PC = +0.130
Arm C (GNN):      PC = 0.3647


In [10]:
# Cell 10 — Load real N-CMAPSS DS02 data

import h5py

DATA_PATH = r'C:\Users\sohan\OneDrive\Desktop\lime-shap-explainability-engineering-failure\data\N-CMAPSS_DS02-006.h5'

with h5py.File(DATA_PATH, 'r') as f:
    print("Keys in file:")
    for key in f.keys():
        print(f"  {key}: {f[key].shape}")

Keys in file:
  A_dev: (5263447, 4)
  A_test: (1253743, 4)
  A_var: (4,)
  T_dev: (5263447, 10)
  T_test: (1253743, 10)
  T_var: (10,)
  W_dev: (5263447, 4)
  W_test: (1253743, 4)
  W_var: (4,)
  X_s_dev: (5263447, 14)
  X_s_test: (1253743, 14)
  X_s_var: (14,)
  X_v_dev: (5263447, 14)
  X_v_test: (1253743, 14)
  X_v_var: (14,)
  Y_dev: (5263447, 1)
  Y_test: (1253743, 1)


In [11]:
# Cell 11 — Load and prepare real N-CMAPSS data

import h5py
import numpy as np

DATA_PATH = r'C:\Users\sohan\OneDrive\Desktop\lime-shap-explainability-engineering-failure\data\N-CMAPSS_DS02-006.h5'

with h5py.File(DATA_PATH, 'r') as f:
    X_s_dev = f['X_s_dev'][:]
    X_s_test = f['X_s_test'][:]
    T_dev = f['T_dev'][:]
    T_test = f['T_test'][:]
    Y_dev = f['Y_dev'][:].flatten()
    Y_test = f['Y_test'][:].flatten()
    A_dev = f['A_dev'][:]
    A_test = f['A_test'][:]

print(f"Training samples: {X_s_dev.shape}")
print(f"Test samples: {X_s_test.shape}")
print(f"Health params shape: {T_dev.shape}")
print(f"RUL range: {Y_dev.min():.1f} to {Y_dev.max():.1f}")
print(f"Unique train engines: {len(np.unique(A_dev[:,0]))}")
print(f"Unique test engines: {len(np.unique(A_test[:,0]))}")
print("Real data loaded successfully")

Training samples: (5263447, 14)
Test samples: (1253743, 14)
Health params shape: (5263447, 10)
RUL range: 0.0 to 88.0
Unique train engines: 6
Unique test engines: 3
Real data loaded successfully


In [ ]:
# Cell 12 — Create windows from real data

from sklearn.preprocessing import StandardScaler

WINDOW = 30
MAX_RUL = 88  # real data max RUL

def make_windows_real(X, Y, T, A, window=WINDOW):
    windows, labels, health_windows = [], [], []
    
    # Get unique engine IDs
    engine_ids = np.unique(A[:, 0])
    
    for eng_id in engine_ids:
        # Get data for this engine
        mask = A[:, 0] == eng_id
        X_eng = X[mask]
        Y_eng = Y[mask]
        T_eng = T[mask]
        
        # Create sliding windows
        for i in range(len(X_eng) - window):
            windows.append(X_eng[i:i+window])
            labels.append(Y_eng[i+window-1])
            health_windows.append(T_eng[i+window-1])
    
    return (np.array(windows, dtype=np.float32),
            np.array(labels, dtype=np.float32),
            np.array(health_windows, dtype=np.float32))

print("Creating windows from real data (this may take a few minutes)...")
X_train_real, Y_train_real, H_train_real = make_windows_real(X_s_dev, Y_dev, T_dev, A_dev)
X_test_real, Y_test_real, H_test_real = make_windows_real(X_s_test, Y_test, T_test, A_test)

print(f"Train windows: {X_train_real.shape}")
print(f"Test windows: {X_test_real.shape}")
print(f"Health params: {H_train_real.shape}")

# Normalise
scaler_real = StandardScaler()
X_train_flat = X_train_real.reshape(-1, 14)
X_train_real_norm = scaler_real.fit_transform(X_train_flat).reshape(X_train_real.shape)
X_test_flat = X_test_real.reshape(-1, 14)
X_test_real_norm = scaler_real.transform(X_test_flat).reshape(X_test_real.shape)

# Normalise RUL
Y_train_real_norm = np.clip(Y_train_real / MAX_RUL, 0, 1)
Y_test_real_norm = np.clip(Y_test_real / MAX_RUL, 0, 1)

# Normalise health params to 0-1
H_train_real_norm = np.clip(H_train_real, 0, 1)
H_test_real_norm = np.clip(H_test_real, 0, 1)

print("Preprocessing complete")

Creating windows from real data (this may take a few minutes)...
Train windows: (5263267, 30, 14)
Test windows: (1253653, 30, 14)
Health params: (5263267, 10)


In [ ]:
# Cell 13 — Prepare GNN dataset from real data
# Using same physics graph structure

import torch
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as GeoDataLoader

SENSOR_COMP_INFLUENCE = np.array([
    [0.8, 0.1, 0.0, 0.0, 0.0],
    [0.1, 0.7, 0.2, 0.0, 0.0],
    [0.0, 0.1, 0.8, 0.1, 0.0],
    [0.0, 0.0, 0.1, 0.8, 0.1],
    [0.6, 0.2, 0.0, 0.0, 0.0],
    [0.5, 0.1, 0.0, 0.0, 0.0],
    [0.3, 0.5, 0.1, 0.0, 0.0],
    [0.0, 0.2, 0.7, 0.1, 0.0],
    [0.0, 0.1, 0.8, 0.1, 0.0],
    [0.0, 0.0, 0.2, 0.7, 0.1],
    [0.0, 0.0, 0.1, 0.2, 0.7],
    [0.7, 0.2, 0.1, 0.0, 0.0],
    [0.1, 0.2, 0.6, 0.1, 0.0],
    [0.0, 0.1, 0.5, 0.3, 0.1],
])

COMPONENTS = ['Fan', 'LPC', 'HPC', 'HPT', 'LPT']

edge_index = torch.tensor([
    [0, 1, 2, 3],
    [1, 2, 3, 4]
], dtype=torch.long)

M_np = SENSOR_COMP_INFLUENCE

# Health params in real data are 10 values (2 per component)
# Average pairs to get 5 concept scores
def get_concepts_from_health(H):
    # H shape: (N, 10) — efficiency + flow for each of 5 components
    concepts = np.zeros((len(H), 5))
    for k in range(5):
        concepts[:, k] = (H[:, 2*k] + H[:, 2*k+1]) / 2
    return concepts.astype(np.float32)

# Subsample to make it manageable — use 10% of data
np.random.seed(42)
N_train = len(X_train_real_norm)
N_test = len(X_test_real_norm)
train_idx = np.random.choice(N_train, size=min(50000, N_train), replace=False)
test_idx = np.random.choice(N_test, size=min(15000, N_test), replace=False)

X_tr = X_train_real_norm[train_idx]
Y_tr = Y_train_real_norm[train_idx]
H_tr = get_concepts_from_health(H_train_real_norm[train_idx])

X_te = X_test_real_norm[test_idx]
Y_te = Y_test_real_norm[test_idx]
H_te = get_concepts_from_health(H_test_real_norm[test_idx])

def prepare_gnn_data_real(X, Y, H):
    dataset = []
    for i in range(len(X)):
        sensor_avg = X[i].mean(axis=0)
        node_features = (M_np.T * sensor_avg).astype(np.float32)
        data = Data(
            x=torch.tensor(node_features, dtype=torch.float32),
            edge_index=edge_index,
            y=torch.tensor([Y[i]], dtype=torch.float32),
            health=torch.tensor(H[i], dtype=torch.float32)
        )
        dataset.append(data)
    return dataset

print("Preparing GNN dataset from real data...")
train_dataset_real = prepare_gnn_data_real(X_tr, Y_tr, H_tr)
test_dataset_real = prepare_gnn_data_real(X_te, Y_te, H_te)

train_loader_real = GeoDataLoader(train_dataset_real, batch_size=256, shuffle=True)
test_loader_real = GeoDataLoader(test_dataset_real, batch_size=256, shuffle=False)

print(f"Train graphs: {len(train_dataset_real)}")
print(f"Test graphs: {len(test_dataset_real)}")
print("Real GNN dataset ready")

Preparing GNN dataset from real data...
Train graphs: 50000
Test graphs: 15000
Real GNN dataset ready


In [ ]:
# Cell 14 — Train Physics GNN on real N-CMAPSS data

import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {DEVICE}")

class PhysicsGNN(nn.Module):
    def __init__(self, n_sensors=14, hidden=64, n_concepts=5):
        super().__init__()
        self.input_proj = nn.Linear(n_sensors, hidden)
        self.conv1 = GCNConv(hidden, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.concept_head = nn.Linear(hidden, 1)
        self.rul_head = nn.Sequential(
            nn.Linear(n_concepts, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        self.n_concepts = n_concepts

    def forward(self, x, edge_index, batch=None):
        h = F.relu(self.input_proj(x))
        h = F.relu(self.conv1(h, edge_index))
        h = F.dropout(h, p=0.2, training=self.training)
        h = F.relu(self.conv2(h, edge_index))
        concepts = torch.sigmoid(self.concept_head(h))
        if batch is not None:
            n_graphs = batch.max().item() + 1
            concepts = concepts.view(n_graphs, self.n_concepts)
        else:
            concepts = concepts.view(-1, self.n_concepts)
        rul = self.rul_head(concepts)
        return rul, concepts

model_real = PhysicsGNN(n_sensors=14, hidden=64, n_concepts=5).to(DEVICE)
optimizer = torch.optim.Adam(model_real.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
rul_loss_fn = nn.MSELoss()
concept_loss_fn = nn.MSELoss()
ALPHA = 0.25

best_val_loss = float('inf')
patience_counter = 0
PATIENCE = 10

print("Training Physics GNN on real N-CMAPSS data...")
print(f"Alpha = {ALPHA}")
print()

for epoch in range(80):
    model_real.train()
    train_losses = []

    for batch in train_loader_real:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()
        rul_pred, concepts = model_real(batch.x, batch.edge_index, batch.batch)
        loss_rul = rul_loss_fn(rul_pred.squeeze(), batch.y.squeeze())
        n_graphs = batch.batch.max().item() + 1
        concept_targets = batch.health.view(n_graphs, 5)
        loss_concept = concept_loss_fn(concepts, concept_targets)
        loss = ALPHA * loss_concept + (1 - ALPHA) * loss_rul
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_real.parameters(), 1.0)
        optimizer.step()
        train_losses.append(loss.item())

    model_real.eval()
    val_losses = []
    with torch.no_grad():
        for batch in test_loader_real:
            batch = batch.to(DEVICE)
            rul_pred, concepts = model_real(batch.x, batch.edge_index, batch.batch)
            loss_rul = rul_loss_fn(rul_pred.squeeze(), batch.y.squeeze())
            val_losses.append(loss_rul.item())

    train_loss = np.mean(train_losses)
    val_loss = np.mean(val_losses)
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model_real.state_dict(), 'best_model_real.pt')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/80 — train: {train_loss:.5f} val: {val_loss:.5f}")

model_real.load_state_dict(torch.load('best_model_real.pt'))
print("\nTraining complete on real data.")

Training on: cpu
Training Physics GNN on real N-CMAPSS data...
Alpha = 0.25

Epoch 10/80 — train: 0.00753 val: 0.00407
Early stopping at epoch 20

Training complete on real data.


In [ ]:
# Cell 15 — Evaluate on real data and compute PC score

from scipy.stats import spearmanr
from sklearn.metrics import mean_squared_error, mean_absolute_error

model_real.eval()
all_preds = []
all_trues = []
all_concepts = []

with torch.no_grad():
    for batch in test_loader_real:
        batch = batch.to(DEVICE)
        rul_pred, concepts = model_real(batch.x, batch.edge_index, batch.batch)
        all_preds.append(rul_pred.squeeze().cpu().numpy())
        all_trues.append(batch.y.squeeze().cpu().numpy())
        all_concepts.append(concepts.cpu().numpy())

pred_real = np.concatenate(all_preds) * MAX_RUL
true_real = np.concatenate(all_trues) * MAX_RUL
concepts_real = np.concatenate(all_concepts)

# RMSE and MAE
rmse_real = np.sqrt(mean_squared_error(true_real, pred_real))
mae_real = mean_absolute_error(true_real, pred_real)
print(f"Arm C (Physics GNN) — Real N-CMAPSS Data")
print(f"RMSE: {rmse_real:.2f} cycles")
print(f"MAE:  {mae_real:.2f} cycles")

# PC Score
FAULT_MODE = 'HPC'
fault_comp = COMPONENTS.index(FAULT_MODE)
e_phys = SENSOR_COMP_INFLUENCE[:, fault_comp]
e_phys = e_phys / e_phys.sum()

# Project concepts to sensor space
attr_real = (M_np @ concepts_real.T).T
attr_real_norm = attr_real / (attr_real.sum(axis=1, keepdims=True) + 1e-10)

# rho_phys
rhos = [spearmanr(attr_real_norm[i], e_phys)[0] for i in range(len(attr_real_norm))]
rho_mean = np.mean(rhos)
rho_std = np.std(rhos)

# delta_cons
pairs = [(s, s2) for s in range(14) for s2 in range(14)
         if abs(e_phys[s] - e_phys[s2]) > 0.05]
violations = []
for i in range(len(attr_real_norm)):
    v = sum(1 for s, s2 in pairs
            if (e_phys[s] > e_phys[s2]) != (attr_real_norm[i,s] > attr_real_norm[i,s2]))
    violations.append(v / len(pairs) if pairs else 0)
delta_cons = np.mean(violations)

sigma_stab = 0.0
pc_real = rho_mean * (1 - delta_cons) * (1 - sigma_stab)

print(f"\nPC Score breakdown:")
print(f"  rho_phys:   {rho_mean:.4f} ± {rho_std:.4f}")
print(f"  delta_cons: {delta_cons:.4f}")
print(f"  sigma_stab: {sigma_stab:.6f}")
print(f"  PC (full):  {pc_real:.4f}")

print(f"\n--- Final Comparison (Real Data) ---")
print(f"Arm A (CNN-LSTM): PC = -0.016  RMSE = 14.32")
print(f"Arm B (CBM):      PC = +0.130  RMSE = 15.67")
print(f"Arm C (GNN):      PC = {pc_real:.4f}  RMSE = {rmse_real:.2f}")

Arm C (Physics GNN) — Real N-CMAPSS Data
RMSE: 5.61 cycles
MAE:  4.13 cycles

PC Score breakdown:
  rho_phys:   0.4916 ± 0.1645
  delta_cons: 0.3554
  sigma_stab: 0.000000
  PC (full):  0.3169

--- Final Comparison (Real Data) ---
Arm A (CNN-LSTM): PC = -0.016  RMSE = 14.32
Arm B (CBM):      PC = +0.130  RMSE = 15.67
Arm C (GNN):      PC = 0.3169  RMSE = 5.61


In [ ]:
# Cell 16 — Ablation: Fully Connected GNN (no physics edges)

# Every node connects to every other node
fc_edges = []
for i in range(5):
    for j in range(5):
        if i != j:
            fc_edges.append([i, j])

fc_edge_index = torch.tensor(fc_edges, dtype=torch.long).T
print(f"Physics edges: {edge_index.shape[1]}")
print(f"Fully connected edges: {fc_edge_index.shape[1]}")

def prepare_gnn_data_fc(X, Y, H):
    dataset = []
    for i in range(len(X)):
        sensor_avg = X[i].mean(axis=0)
        node_features = (M_np.T * sensor_avg).astype(np.float32)
        data = Data(
            x=torch.tensor(node_features, dtype=torch.float32),
            edge_index=fc_edge_index,
            y=torch.tensor([Y[i]], dtype=torch.float32),
            health=torch.tensor(H[i], dtype=torch.float32)
        )
        dataset.append(data)
    return dataset

print("Preparing fully connected GNN dataset...")
train_dataset_fc = prepare_gnn_data_fc(X_tr, Y_tr, H_tr)
test_dataset_fc = prepare_gnn_data_fc(X_te, Y_te, H_te)

train_loader_fc = GeoDataLoader(train_dataset_fc, batch_size=256, shuffle=True)
test_loader_fc = GeoDataLoader(test_dataset_fc, batch_size=256, shuffle=False)

print(f"Train graphs: {len(train_dataset_fc)}")
print(f"Test graphs: {len(test_dataset_fc)}")
print("Fully connected dataset ready")

Physics edges: 4
Fully connected edges: 20
Preparing fully connected GNN dataset...
Train graphs: 50000
Test graphs: 15000
Fully connected dataset ready


In [ ]:
# Cell 17 — Train Fully Connected GNN (ablation)

model_fc = PhysicsGNN(n_sensors=14, hidden=64, n_concepts=5).to(DEVICE)
optimizer_fc = torch.optim.Adam(model_fc.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler_fc = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_fc, patience=5, factor=0.5)

best_val_loss_fc = float('inf')
patience_counter_fc = 0
PATIENCE = 10

print("Training Fully Connected GNN (ablation)...")
print()

for epoch in range(80):
    model_fc.train()
    train_losses = []

    for batch in train_loader_fc:
        batch = batch.to(DEVICE)
        optimizer_fc.zero_grad()
        rul_pred, concepts = model_fc(batch.x, batch.edge_index, batch.batch)
        loss_rul = rul_loss_fn(rul_pred.squeeze(), batch.y.squeeze())
        n_graphs = batch.batch.max().item() + 1
        concept_targets = batch.health.view(n_graphs, 5)
        loss_concept = concept_loss_fn(concepts, concept_targets)
        loss = ALPHA * loss_concept + (1 - ALPHA) * loss_rul
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_fc.parameters(), 1.0)
        optimizer_fc.step()
        train_losses.append(loss.item())

    model_fc.eval()
    val_losses = []
    with torch.no_grad():
        for batch in test_loader_fc:
            batch = batch.to(DEVICE)
            rul_pred, concepts = model_fc(batch.x, batch.edge_index, batch.batch)
            loss_rul = rul_loss_fn(rul_pred.squeeze(), batch.y.squeeze())
            val_losses.append(loss_rul.item())

    train_loss = np.mean(train_losses)
    val_loss = np.mean(val_losses)
    scheduler_fc.step(val_loss)

    if val_loss < best_val_loss_fc:
        best_val_loss_fc = val_loss
        torch.save(model_fc.state_dict(), 'best_model_fc.pt')
        patience_counter_fc = 0
    else:
        patience_counter_fc += 1
        if patience_counter_fc >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/80 — train: {train_loss:.5f} val: {val_loss:.5f}")

model_fc.load_state_dict(torch.load('best_model_fc.pt'))
print("\nFully connected GNN training complete.")

Training Fully Connected GNN (ablation)...

Epoch 10/80 — train: 0.00657 val: 0.00596
Epoch 20/80 — train: 0.00530 val: 0.00395
Early stopping at epoch 30

Fully connected GNN training complete.


In [ ]:
# Cell 18 — Evaluate Fully Connected GNN and compare

model_fc.eval()
all_preds_fc = []
all_trues_fc = []
all_concepts_fc = []

with torch.no_grad():
    for batch in test_loader_fc:
        batch = batch.to(DEVICE)
        rul_pred, concepts = model_fc(batch.x, batch.edge_index, batch.batch)
        all_preds_fc.append(rul_pred.squeeze().cpu().numpy())
        all_trues_fc.append(batch.y.squeeze().cpu().numpy())
        all_concepts_fc.append(concepts.cpu().numpy())

pred_fc = np.concatenate(all_preds_fc) * MAX_RUL
true_fc = np.concatenate(all_trues_fc) * MAX_RUL
concepts_fc = np.concatenate(all_concepts_fc)

rmse_fc = np.sqrt(mean_squared_error(true_fc, pred_fc))
mae_fc = mean_absolute_error(true_fc, pred_fc)

# PC Score for fully connected GNN
attr_fc = (M_np @ concepts_fc.T).T
attr_fc_norm = attr_fc / (attr_fc.sum(axis=1, keepdims=True) + 1e-10)

rhos_fc = [spearmanr(attr_fc_norm[i], e_phys)[0] for i in range(len(attr_fc_norm))]
rho_mean_fc = np.mean(rhos_fc)

violations_fc = []
for i in range(len(attr_fc_norm)):
    v = sum(1 for s, s2 in pairs
            if (e_phys[s] > e_phys[s2]) != (attr_fc_norm[i,s] > attr_fc_norm[i,s2]))
    violations_fc.append(v / len(pairs) if pairs else 0)
delta_cons_fc = np.mean(violations_fc)

pc_fc = rho_mean_fc * (1 - delta_cons_fc)

print("=== Ablation Results ===")
print(f"\nFully Connected GNN:")
print(f"  RMSE: {rmse_fc:.2f} cycles")
print(f"  rho_phys: {rho_mean_fc:.4f}")
print(f"  delta_cons: {delta_cons_fc:.4f}")
print(f"  PC score: {pc_fc:.4f}")

print(f"\n=== Full Comparison Table ===")
print(f"{'Model':<25} {'RMSE':>6} {'PC Score':>10}")
print(f"{'-'*45}")
print(f"{'Arm A (CNN-LSTM)':<25} {'14.32':>6} {'-0.016':>10}")
print(f"{'Arm B (CBM)':<25} {'15.67':>6} {'+0.130':>10}")
print(f"{'Arm C-FC (no physics)':<25} {rmse_fc:>6.2f} {pc_fc:>10.4f}")
print(f"{'Arm C (Physics GNN)':<25} {'5.60':>6} {'0.2736':>10}")
print(f"\nPhysics edges vs fully connected: PC {0.2736:.4f} vs {pc_fc:.4f}")
if 0.2736 > pc_fc:
    print("Physics edges WIN — graph structure is doing the work")
else:
    print("Interesting — fully connected scores higher, investigate further")

=== Ablation Results ===

Fully Connected GNN:
  RMSE: 5.53 cycles
  rho_phys: 0.7185
  delta_cons: 0.2540
  PC score: 0.5360

=== Full Comparison Table ===
Model                       RMSE   PC Score
---------------------------------------------
Arm A (CNN-LSTM)           14.32     -0.016
Arm B (CBM)                15.67     +0.130
Arm C-FC (no physics)       5.53     0.5360
Arm C (Physics GNN)         5.60     0.2736

Physics edges vs fully connected: PC 0.2736 vs 0.5360
Interesting — fully connected scores higher, investigate further
